In [ ]:
%pip install --quiet --upgrade diffusers transformers accelerate mediapy peft

**The primary reason LoRA (Low-Rank Adaptation) is used in this context is to make it feasible to run the fine-tuning or adapted model on resource-constrained environments, such as Google Colab (or other setups with limited GPU memory).**

In [ ]:
import mediapy as media
import random
import sys
import torch

from diffusers import DiffusionPipeline, TCDScheduler
from huggingface_hub import hf_hub_download

# Choose either 8 or 12 steps for the diffusion process:
# This affects the tradeoff between generation speed and image quality.
num_inference_steps = 12

# Base model: Stable Diffusion XL (SDXL), a high-capacity image synthesis model.
base_model_id = "stabilityai/stable-diffusion-xl-base-1.0"

# Hugging Face repository containing LoRA weights for the model.
repo_name = "ByteDance/Hyper-SD"

# Adjust checkpoint name based on the number of inference steps.
plural = "s" if num_inference_steps > 1 else ""
ckpt_name = f"Hyper-SDXL-{num_inference_steps}step{plural}-CFG-lora.safetensors"

# Use GPU for computation if available (CUDA).
device = "cuda"

# Initialize the Stable Diffusion pipeline with the base model.
# - torch_dtype=torch.float16: Use 16-bit precision for faster computation and reduced memory usage.
# - variant="fp16": Ensures compatibility with 16-bit precision.
pipe = DiffusionPipeline.from_pretrained(base_model_id, torch_dtype=torch.float16, variant="fp16").to(device)

# Download the LoRA (Low-Rank Adaptation) checkpoint from the Hugging Face Hub.
# LoRA is a method for fine-tuning large models efficiently:
# - Instead of retraining all model parameters, LoRA updates only a small subset of parameters
#   (low-rank matrices added to certain layers). This reduces computational cost and memory usage.
# - LoRA allows a base model to be quickly adapted to specific tasks or styles.
pipe.load_lora_weights(hf_hub_download(repo_name, ckpt_name))

# Fuse the LoRA weights into the model, making it ready for inference.
pipe.fuse_lora()

# Replace the default scheduler with TCDScheduler, which guides the diffusion process.
# Schedulers determine how the generated image is refined step-by-step.
pipe.scheduler = TCDScheduler.from_config(pipe.scheduler.config)


In [ ]:
prompt = "frog in hut raining outside"
seed = random.randint(0, sys.maxsize)

# Pick a value between 5.0 and 8.0:
guidance_scale = 5.0

# Decrease eta (min: 0, max: 1.0) to get more details with multi-step inference:
eta = 0

images = pipe(
    prompt = prompt,
    num_inference_steps = num_inference_steps,
    guidance_scale = guidance_scale,
    eta = eta,
    generator = torch.Generator(device).manual_seed(seed),
    ).images

print(f"Prompt:\t{prompt}\nSeed:\t{seed}")
media.show_images(images)
images[0].save("output.jpg")

In [ ]:
images[0]